In [12]:
# Práctica 8: Forecasting Create a model using linear regression and predict new data. Use a time series.
# Se estará usando el dataset original sin imputación simplemente haciendo una limpieza de datos para ver si el rendimiento mejora en comparación de prácticas anteriores

import pandas as pd

df = pd.read_csv("ncr_ride_bookings.csv")
print(df.head())

print("Null data before cleaning: \n", df.isna().sum())
df['Reason for cancelling by Customer'] = df['Reason for cancelling by Customer'].fillna('Not cancelled')
df['Cancelled Rides by Driver'] = df['Cancelled Rides by Driver'].fillna(0)
df['Cancelled Rides by Customer'] = df['Cancelled Rides by Customer'].fillna(0)
df['Driver Cancellation Reason'] = df['Driver Cancellation Reason'].fillna('Not cancelled')
df['Incomplete Rides'] = df['Incomplete Rides'].fillna(0)
df['Incomplete Rides Reason'] = df['Incomplete Rides Reason'].fillna('Not incomplete')

df['Booking Value'] = df['Booking Value'].fillna(0)
df['Ride Distance'] = df['Ride Distance'].fillna(0)
df['Driver Ratings'] = df['Driver Ratings'].fillna(0)
df['Customer Rating'] = df['Customer Rating'].fillna(0)
df['Avg CTAT'] = df['Avg CTAT'].fillna(0)
df['Avg VTAT'] = df['Avg VTAT'].fillna(0)
df['Payment Method'] = df['Payment Method'].fillna('Cancelled or incomplete ride')

df['is_cancelled'] = (df['Cancelled Rides by Customer'] == 1) | (df['Cancelled Rides by Driver'] == 1)
df['is_cancelled'] = df['is_cancelled'].astype(int)

print("\n Null data after cleaning: \n", df.isna().sum())

         Date      Time    Booking ID   Booking Status   Customer ID  \
0  2024-03-23  12:29:38  "CNR5884300"  No Driver Found  "CID1982111"   
1  2024-11-29  18:01:39  "CNR1326809"       Incomplete  "CID4604802"   
2  2024-08-23  08:56:10  "CNR8494506"        Completed  "CID9202816"   
3  2024-10-21  17:17:25  "CNR8906825"        Completed  "CID2610914"   
4  2024-09-16  22:08:00  "CNR1950162"        Completed  "CID9933542"   

    Vehicle Type      Pickup Location      Drop Location  Avg VTAT  Avg CTAT  \
0          eBike          Palam Vihar            Jhilmil       NaN       NaN   
1       Go Sedan        Shastri Nagar  Gurgaon Sector 56       4.9      14.0   
2           Auto              Khandsa      Malviya Nagar      13.4      25.8   
3  Premier Sedan  Central Secretariat           Inderlok      13.1      28.5   
4           Bike     Ghitorni Village        Khan Market       5.3      19.6   

   ...  Reason for cancelling by Customer Cancelled Rides by Driver  \
0  ...         

In [13]:
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df = df.set_index('DateTime')
df = df.sort_index()

daily_bookings = df.resample('D').size().reset_index(name='booking_count')
daily_bookings = daily_bookings.set_index('DateTime')

print("Original DataFrame head:")
print(df.head())
print("\nDaily Bookings DataFrame head:")
print(daily_bookings.head())

Original DataFrame head:
                           Date      Time    Booking ID       Booking Status  \
DateTime                                                                       
2024-01-01 05:02:14  2024-01-01  05:02:14  "CNR4302396"  Cancelled by Driver   
2024-01-01 05:52:06  2024-01-01  05:52:06  "CNR2972763"  Cancelled by Driver   
2024-01-01 09:45:42  2024-01-01  09:45:42  "CNR4962670"  Cancelled by Driver   
2024-01-01 10:33:44  2024-01-01  10:33:44  "CNR8218840"  Cancelled by Driver   
2024-01-01 11:33:13  2024-01-01  11:33:13  "CNR9442408"            Completed   

                      Customer ID Vehicle Type Pickup Location  \
DateTime                                                         
2024-01-01 05:02:14  "CID3437829"         Auto          Jasola   
2024-01-01 05:52:06  "CID4100766"     Go Sedan     DLF Phase 3   
2024-01-01 09:45:42  "CID3210772"         Bike      Green Park   
2024-01-01 10:33:44  "CID6178341"     Go Sedan        Ghitorni   
2024-01-01 11:33:1

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

daily_bookings['day_ordinal'] = (daily_bookings.index - daily_bookings.index.min()).days

if 'day_of_week' not in daily_bookings.columns:
    daily_bookings['day_of_week'] = daily_bookings.index.dayofweek
    daily_bookings['month_of_year'] = daily_bookings.index.month
    daily_bookings['day_of_month'] = daily_bookings.index.day
    daily_bookings['week_of_year'] = daily_bookings.index.isocalendar().week.astype(int)

X = daily_bookings[['day_ordinal', 'day_of_week', 'month_of_year', 'day_of_month', 'week_of_year']]
y = daily_bookings['booking_count']

train_size = int(len(daily_bookings) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

predictions_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}, index=y_test.index)
print("\nSample of Actual vs Predicted bookings:")
print(predictions_df.head())

last_day_ordinal = daily_bookings['day_ordinal'].max()
future_days_ordinal = np.arange(last_day_ordinal + 1, last_day_ordinal + 31)
future_dates = pd.date_range(start=daily_bookings.index.max() + pd.Timedelta(days=1), periods=30, freq='D')

future_df = pd.DataFrame({
    'day_ordinal': future_days_ordinal,
    'day_of_week': future_dates.dayofweek,
    'month_of_year': future_dates.month,
    'day_of_month': future_dates.day,
    'week_of_year': future_dates.isocalendar().week.astype(int)
}, index=future_dates)

future_predictions = model.predict(future_df)

future_predictions_df = pd.DataFrame({
    'Predicted_Bookings': future_predictions
}, index=future_dates)

print("\nForecasted bookings for the next 30 days:")
print(future_predictions_df.head())

Mean Squared Error (MSE): 13.01
Root Mean Squared Error (RMSE): 3.61

Sample of Actual vs Predicted bookings:
            Actual  Predicted
DateTime                     
2024-10-19      15  17.108547
2024-10-20      15  17.152581
2024-10-21      14  17.028214
2024-10-22      12  17.072248
2024-10-23      13  17.116283

Forecasted bookings for the next 30 days:
            Predicted_Bookings
2024-12-31           15.235468
2025-01-01          128.319006
2025-01-02          128.363040
2025-01-03          128.407075
2025-01-04          128.451109
